# Synthetic Datasets: Plots of Individual Fairness Metrics

Author: Ilse Harmers \
Last modified: March 17, 2026

Certain sections of code in this notebook have been kindly provided by dr. Mina Alishahi.

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import ticker
import matplotlib
params = {'axes.titlesize':'18',
          'xtick.labelsize':'15',
          'ytick.labelsize':'15',
          'font.size':'16',
          'legend.fontsize':'medium',
          'lines.linewidth':'2.0',
          'font.weight':'normal',
          'lines.markersize':'5',
          'text.latex.preamble': r'\usepackage{amsfonts}',
          }
matplotlib.rcParams.update(params)
plt.rcParams["mathtext.fontset"] = "cm"
plt.rc('text', usetex=True)
plt.rc('font', family='serif')

from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

## Initialization

In [ ]:
# Initializing the real datasets (Adult, Bank Marketing and Credit Card Default), our classifier and our functions. 
adult_train_encoded = pd.read_csv("./train-test-datasets/adult/train_encoded.csv")
adult_test_encoded = pd.read_csv("./train-test-datasets/adult/test_encoded.csv")
bank_train_encoded = pd.read_csv("./train-test-datasets/bank-marketing/train_encoded.csv")
bank_test_encoded = pd.read_csv("./train-test-datasets/bank-marketing/test_encoded.csv")
credit_train_encoded = pd.read_csv("./train-test-datasets/credit-card-default/train_encoded.csv")
credit_test_encoded = pd.read_csv("./train-test-datasets/credit-card-default/test_encoded.csv")

# Initializing our classifiers.
rf_clf = RandomForestClassifier()
mlp_clf = MLPClassifier(max_iter = 500)
lda_clf = LinearDiscriminantAnalysis()

classifiers = [rf_clf, mlp_clf, lda_clf]

def encode_adult(sample):
    """This function encodes an Adult-based synthetic dataset such that all ordinal columns are scaled to a range of (0, 1)."""

    sample["education-num"] = ((sample["education-num"] - sample["education-num"].min()) / 
                               (sample["education-num"].max() - sample["education-num"].min()))

    return sample

def encode_bank(sample):
    """This function encodes a Bank-based synthetic dataset such that all ordinal columns are scaled to a range of (0, 1)."""

    sample["day"] = ((sample["day"] - sample["day"].min()) / (sample["day"].max() - sample["day"].min()))
    
    return sample
    
def encode_credit(sample):
    """This function encodes a Credit-based synthetic dataset such that all ordinal columns are scaled to a range of (0, 1)."""

    sample["SEX"] = ((sample["SEX"] - sample["SEX"].min()) / (sample["SEX"].max() - sample["SEX"].min()))
    
    for p in [0, 2, 3, 4, 5, 6]:
        sample[f"PAY_{p}"] = ((sample[f"PAY_{p}"] - sample[f"PAY_{p}"].min()) / 
                              (sample[f"PAY_{p}"].max() - sample[f"PAY_{p}"].min()))

    return sample

def compute_similarity_fairness(indices, predictions, distances, neighbors):
    """
    Compute individual fairness for a chunk of data using precomputed k-NN distances and neighbors.

    Parameters:
        indices (list): Indices of the chunk.
        predictions (array-like): Model predictions for individuals.
        distances (array-like): Distances from k-NN.
        neighbors (array-like): Indices of k-NN neighbors.

    Returns:
        float: Mean individual fairness score for the chunk.

    Code provided by: dr. Mina Alishahi
    """
    
    fairness_scores = []
    for idx in indices:
        # Iterate through the k-nearest neighbors (skip the first as it's the individual itself).
        for neighbor_idx, distance in zip(neighbors[idx][1:], distances[idx][1:]):
            if distance < 1e-6:  # Treat very small distances as non-zero.
                distance = 1e-6
            pred_diff = abs(predictions[idx] - predictions[neighbor_idx])
            fairness_scores.append(pred_diff * distance)
    return np.mean(fairness_scores) if fairness_scores else 0.0

def similarity_fairness(predictions, features, similarity_metric = "euclidean", k = 100, num_chunks = 8):
    """
    Compute individual fairness for model predictions using k-NN.

    Parameters:
        predictions (array-like): Model predictions for individuals.
        features (array-like): Feature matrix (N x D).
        similarity_metric (str): Similarity metric ('cosine' or 'euclidean').
        k (int): Number of nearest neighbors to consider.
        num_chunks (int): Number of chunks.

    Returns:
        float: Average individual fairness score (lower is better).

    Code provided by: dr. Mina Alishahi
    """
    
    n = len(features)
    if k >= n:
        raise ValueError(f"Invalid k: {k}. Must be less than the number of data points: {n}.")
    
    # Fit k-NN on the features.
    np.random.seed(42)   # setting random seed
    knn = NearestNeighbors(n_neighbors = k + 1, metric = similarity_metric).fit(features)
    distances, neighbors = knn.kneighbors(features, return_distance=True)

    # Divide indices into chunks.
    chunk_size = max(1, n // num_chunks)
    fairness_scores = [
        compute_similarity_fairness(
            range(start, min(start + chunk_size, n)), predictions, distances, neighbors
        )
        for start in range(0, n, chunk_size)
    ]

    # Gather results from all chunks and compute the overall mean.
    return np.mean(fairness_scores) if fairness_scores else 0.0


def compute_neighborhood_consistency_fairness(indices, predictions, distances, neighbors):
    """
    Compute neighborhood consistency for a chunk of data.

    Parameters:
        indices (list): Indices of the chunk.
        predictions (array-like): Model predictions for individuals.
        distances (array-like): Distances from k-NN.
        neighbors (array-like): Indices of k-NN neighbors.

    Returns:
        float: Mean neighborhood consistency score for the chunk.

    Code provided by: dr. Mina Alishahi
    """
    consistency_scores = []
    for idx in indices:
        # Calculate consistency score for the current individual.
        local_consistency = np.mean([
            abs(predictions[idx] - predictions[neighbor_idx])
            for neighbor_idx in neighbors[idx][1:]  # Skip the first neighbor (the individual itself).
        ])
        consistency_scores.append(local_consistency)

    return np.mean(consistency_scores)

def neighborhood_consistency_fairness(predictions, features, similarity_metric = "euclidean", k = 100, num_chunks = 8):
    """
    Compute neighborhood consistency metric using k-NN.

    Parameters:
        predictions (array-like): Model predictions for individuals.
        features (array-like): Feature matrix (N x D).
        similarity_metric (str): Similarity metric ('cosine' or 'euclidean').
        k (int): Number of nearest neighbors to consider.
        num_chunks (int): Number of chunks.

    Returns:
        float: Average neighborhood consistency score (lower is better).

    Code provided by: dr. Mina Alishahi
    """
        
    # Fit k-NN on the features.
    np.random.seed(42)   # setting random seed
    knn = NearestNeighbors(n_neighbors = k + 1, metric = similarity_metric).fit(features)
    distances, neighbors = knn.kneighbors(features, return_distance = True)

    # Divide indices into chunks.
    n = len(features)
    chunk_size = n // num_chunks
    consistency_scores = [
        compute_neighborhood_consistency_fairness(
            range(start, min(start + chunk_size, n)), predictions, distances, neighbors
        )
        for start in range(0, n, chunk_size)
    ]
    
    # Gather results from all chunks and compute the overall mean.
    return np.mean(consistency_scores)

def fairness_analysis(train_set, test_set, dataset):
    """This function computes the similarity fairness and neigborhood consistency fairness scores for a given train and test set.
    
    train_set [array-like]: train set 
    test_set [array-like]: test set
    dataset [string]: dataset name; can be set to "Adult", "Bank" or "Credit"
    """

    if dataset == "Adult":
        y = "income"
    elif dataset == "Bank":
        y = "y"
    elif dataset == "Credit":
        y = "DEFAULT"
    else:
        raise ValueError("Dataset not supported.")
        
    asf_scores = []   # similarity fairness
    ncf_scores = []   # neighborhood consistency fairness
    # Computing the similarity fairness and neighborhood consistency fairness scores. 
    for clf in classifiers:
        np.random.seed(42)   # setting random seed
        model_real = clf.fit(train_set.drop(columns = [y]), train_set[y])
        soft_ypred = model_real.predict_proba(test_set.drop(columns = [y]))
        ypred = model_real.predict(test_set.drop(columns = [y]))

        # Min-max scaling the ordinal columns.
        if dataset == "Adult":
            test_copy = test_set.copy()
            test_enc = encode_adult(test_copy)
        elif dataset == "Bank":
            test_copy = test_set.copy()
            test_enc = encode_bank(test_copy)
        elif dataset == "Credit":
            test_copy = test_set.copy()
            test_enc = encode_credit(test_copy)
        else:
            raise ValueError("Dataset not supported.")
    
        asf_score = similarity_fairness(soft_ypred, test_enc.drop(columns = [y]).values)   # similarity fairness
        asf_scores.append(np.abs(asf_score))

        ncf_score = neighborhood_consistency_fairness(ypred, test_enc.drop(columns = [y]).values)   # neighborhood consistency fairness
        ncf_scores.append(np.abs(ncf_score))

    return (asf_scores, ncf_scores)

## Adult Dataset (Real)

In [ ]:
# Individual fairness analysis of Adult dataset.
adult_asf, adult_ncf = fairness_analysis(train_set = adult_train_encoded, test_set = adult_test_encoded, dataset = "Adult")
print("asf_scores (mean, std): ", (np.mean(adult_asf), np.std(adult_asf)))  
print("ncf_scores (mean, std): ", (np.mean(adult_ncf), np.std(adult_ncf)))

adult_results = pd.DataFrame({"asf": (np.mean(adult_asf), np.std(adult_asf)), 
                              "ncf": (np.mean(adult_ncf), np.std(adult_ncf))}).rename(index = {0: "mean", 1: "std"})

In [ ]:
print(adult_asf, adult_ncf)
adult_results

## Synthetic Adult Dataset (PATE-GAN)

In [ ]:
# Assigning file path for importing the datasets.
file_path = "./synthetic-datasets_PATE-GAN/adult/"

# Computing the metrics for the best performing datasets.
sample_11a = pd.read_csv(file_path + "epsi_1/run4/sample2_encoded.csv")
asf_11a, ncf_11a = fairness_analysis(train_set = sample_11a, test_set = adult_test_encoded, dataset = "Adult")
print("asf_scores (mean, std): ", (np.mean(asf_11a), np.std(asf_11a)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_11a), np.std(ncf_11a)))
print()

sample_11b = pd.read_csv(file_path + "epsi_2/run4/sample2_encoded.csv")
asf_11b, ncf_11b = fairness_analysis(train_set = sample_11b, test_set = adult_test_encoded, dataset = "Adult")
print("asf_scores (mean, std): ", (np.mean(asf_11b), np.std(asf_11b)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_11b), np.std(ncf_11b)))
print()

sample_11c = pd.read_csv(file_path + "epsi_5/run4/sample2_encoded.csv")
asf_11c, ncf_11c = fairness_analysis(train_set = sample_11c, test_set = adult_test_encoded, dataset = "Adult")
print("asf_scores (mean, std): ", (np.mean(asf_11c), np.std(asf_11c)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_11c), np.std(ncf_11c)))
print()

sample_13 = pd.read_csv(file_path + "epsi_8/run5/sample1_encoded.csv")
asf_13, ncf_13 = fairness_analysis(train_set = sample_13, test_set = adult_test_encoded, dataset = "Adult")
print("asf_scores (mean, std): ", (np.mean(asf_13), np.std(asf_13)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_13), np.std(ncf_13)))

adult_results_PT = pd.DataFrame({"sample_11a": {"asf": np.mean(asf_11a), "ncf": np.mean(ncf_11a)}, 
                                 "sample_11b": {"asf": np.mean(asf_11b), "ncf": np.mean(ncf_11b)}, 
                                 "sample_11c": {"asf": np.mean(asf_11c), "ncf": np.mean(ncf_11c)},
                                 "sample_13": {"asf": np.mean(asf_13), "ncf": np.mean(ncf_13)}}).T
adult_results_PT_std = pd.DataFrame({"sample_11a": {"asf": np.std(asf_11a), "ncf": np.std(ncf_11a)}, 
                                     "sample_11b": {"asf": np.std(asf_11b), "ncf": np.std(ncf_11b)}, 
                                     "sample_11c": {"asf": np.std(asf_11c), "ncf": np.std(ncf_11c)},
                                     "sample_13": {"asf": np.std(asf_13), "ncf": np.std(ncf_13)}}).T

adult_results_PT_full = pd.DataFrame({"sample_11a": {"asf": asf_11a, "ncf": ncf_11a}, 
                                      "sample_11b": {"asf": asf_11b, "ncf": ncf_11b}, 
                                      "sample_11c": {"asf": asf_11c, "ncf": ncf_11c},
                                      "sample_13": {"asf": asf_13, "ncf": ncf_13}}).T

In [ ]:
#adult_results_PT_full["asf"].iloc[3]
#adult_results_PT_full["ncf"].iloc[3]
adult_results_PT_full

## Synthetic Adult Dataset (DP-GAN)

In [ ]:
# Assigning file path for importing the datasets.
file_path = "./synthetic-datasets_DP-GAN_B=512/adult/"

# Computing the metrics for the best performing datasets.
sample_7a = pd.read_csv(file_path + "epsi_1/run3/sample1_encoded.csv")
asf_7a, ncf_7a = fairness_analysis(train_set = sample_7a, test_set = adult_test_encoded, dataset = "Adult")
print("asf_scores (mean, std): ", (np.mean(asf_7a), np.std(asf_7a)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_7a), np.std(ncf_7a)))
print()

sample_7b = pd.read_csv(file_path + "epsi_2/run3/sample1_encoded.csv")
asf_7b, ncf_7b = fairness_analysis(train_set = sample_7b, test_set = adult_test_encoded, dataset = "Adult")
print("asf_scores (mean, std): ", (np.mean(asf_7b), np.std(asf_7b)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_7b), np.std(ncf_7b)))
print()

sample_4 = pd.read_csv(file_path + "epsi_5/run2/sample1_encoded.csv")
asf_4, ncf_4 = fairness_analysis(train_set = sample_4, test_set = adult_test_encoded, dataset = "Adult")
print("asf_scores (mean, std): ", (np.mean(asf_4), np.std(asf_4)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_4), np.std(ncf_4)))
print()

sample_15 = pd.read_csv(file_path + "epsi_8/run5/sample3_encoded.csv")
asf_15, ncf_15 = fairness_analysis(train_set = sample_15, test_set = adult_test_encoded, dataset = "Adult")
print("asf_scores (mean, std): ", (np.mean(asf_15), np.std(asf_15)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_15), np.std(ncf_15)))

adult_results_DP = pd.DataFrame({"sample_7a": {"asf": np.mean(asf_7a), "ncf": np.mean(ncf_7a)}, 
                                 "sample_7b": {"asf": np.mean(asf_7b), "ncf": np.mean(ncf_7b)}, 
                                 "sample_4": {"asf": np.mean(asf_4), "ncf": np.mean(ncf_4)},
                                 "sample_15": {"asf": np.mean(asf_15), "ncf": np.mean(ncf_15)}}).T
adult_results_DP_std = pd.DataFrame({"sample_7a": {"asf": np.std(asf_7a), "ncf": np.std(ncf_7a)}, 
                                     "sample_7b": {"asf": np.std(asf_7b), "ncf": np.std(ncf_7b)}, 
                                     "sample_4": {"asf": np.std(asf_4), "ncf": np.std(ncf_4)},
                                     "sample_15": {"asf": np.std(asf_15), "ncf": np.std(ncf_15)}}).T

adult_results_DP_full = pd.DataFrame({"sample_7a": {"asf": asf_7a, "ncf": ncf_7a}, 
                                      "sample_7b": {"asf": asf_7b, "ncf": ncf_7b}, 
                                      "sample_4": {"asf": asf_4, "ncf": ncf_4},
                                      "sample_15": {"asf": asf_15, "ncf": ncf_15}}).T

In [ ]:
#adult_results_DP_full["asf"].iloc[3]
#adult_results_DP_full["ncf"].iloc[3]
adult_results_DP_full

## Synthetic Adult Dataset (WGAN-GP)

In [ ]:
# Assigning file path for importing the dataset. 
file_path = "./synthetic-datasets_WGAN-GP/adult/"

# Computing the metrics for the best performing dataset.
sample_5 = pd.read_csv(file_path + "run2/sample2_encoded.csv")
adult_wgangp_asf, adult_wgangp_ncf = fairness_analysis(train_set = sample_5, test_set = adult_test_encoded, dataset = "Adult")
print("asf_scores (mean, std): ", (np.mean(adult_wgangp_asf), np.std(adult_wgangp_asf)))  
print("ncf_scores (mean, std): ", (np.mean(adult_wgangp_ncf), np.std(adult_wgangp_ncf)))

adult_wgangp_results = pd.DataFrame({"asf": (np.mean(adult_wgangp_asf), np.std(adult_wgangp_asf)), 
                                     "ncf": (np.mean(adult_wgangp_ncf), np.std(adult_wgangp_ncf))}).rename(index = {0: "mean", 1: "std"})

In [ ]:
print(adult_wgangp_asf, adult_wgangp_ncf)
adult_wgangp_results

## Bank Marketing Dataset (Real)

In [ ]:
# Individual fairness analysis of Bank Marketing dataset.
bank_asf, bank_ncf = fairness_analysis(train_set = bank_train_encoded, test_set = bank_test_encoded, dataset = "Bank")
print("asf_scores (mean, std): ", (np.mean(bank_asf), np.std(bank_asf)))  
print("ncf_scores (mean, std): ", (np.mean(bank_ncf), np.std(bank_ncf)))

bank_results = pd.DataFrame({"asf": (np.mean(bank_asf), np.std(bank_asf)), 
                             "ncf": (np.mean(bank_ncf), np.std(bank_ncf))}).rename(index = {0: "mean", 1: "std"})

In [ ]:
print(bank_asf, bank_ncf)
bank_results 

## Synthetic Bank Marketing Dataset (PATE-GAN)

In [ ]:
# Assigning file path for importing the datasets.
file_path = "./synthetic-datasets_PATE-GAN/bank-marketing/"

# Computing the metrics for the best performing datasets.
sample_2 = pd.read_csv(file_path + "epsi_1/run1/sample2_encoded.csv")
asf_2, ncf_2 = fairness_analysis(train_set = sample_2, test_set = bank_test_encoded, dataset = "Bank")
print("asf_scores (mean, std): ", (np.mean(asf_2), np.std(asf_2)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_2), np.std(ncf_2)))
print()

sample_13 = pd.read_csv(file_path + "epsi_2/run5/sample1_encoded.csv")
asf_13, ncf_13 = fairness_analysis(train_set = sample_13, test_set = bank_test_encoded, dataset = "Bank")
print("asf_scores (mean, std): ", (np.mean(asf_13), np.std(asf_13)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_13), np.std(ncf_13)))
print()

sample_7 = pd.read_csv(file_path + "epsi_5/run3/sample1_encoded.csv")
asf_7, ncf_7 = fairness_analysis(train_set = sample_7, test_set = bank_test_encoded, dataset = "Bank")
print("asf_scores (mean, std): ", (np.mean(asf_7), np.std(asf_7)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_7), np.std(ncf_7)))
print()

sample_11 = pd.read_csv(file_path + "epsi_8/run4/sample2_encoded.csv")
asf_11, ncf_11 = fairness_analysis(train_set = sample_11, test_set = bank_test_encoded, dataset = "Bank")
print("asf_scores (mean, std): ", (np.mean(asf_11), np.std(asf_11)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_11), np.std(ncf_11)))

bank_results_PT = pd.DataFrame({"sample_2": {"asf": np.mean(asf_2), "ncf": np.mean(ncf_2)}, 
                                "sample_13": {"asf": np.mean(asf_13), "ncf": np.mean(ncf_13)}, 
                                "sample_7": {"asf": np.mean(asf_7), "ncf": np.mean(ncf_7)},
                                "sample_11": {"asf": np.mean(asf_11), "ncf": np.mean(ncf_11)}}).T
bank_results_PT_std = pd.DataFrame({"sample_2": {"asf": np.std(asf_2), "ncf": np.std(ncf_2)}, 
                                    "sample_13": {"asf": np.std(asf_13), "ncf": np.std(ncf_13)}, 
                                    "sample_7": {"asf": np.std(asf_7), "ncf": np.std(ncf_7)},
                                    "sample_11": {"asf": np.std(asf_11), "ncf": np.std(ncf_11)}}).T

bank_results_PT_full = pd.DataFrame({"sample_2": {"asf": asf_2, "ncf": ncf_2}, 
                                     "sample_13": {"asf": asf_13, "ncf": ncf_13}, 
                                     "sample_7": {"asf": asf_7, "ncf": ncf_7},
                                     "sample_11": {"asf": asf_11, "ncf": ncf_11}}).T

In [ ]:
#bank_results_PT_full["asf"].iloc[3]
#bank_results_PT_full["ncf"].iloc[3]
bank_results_PT_full

## Synthetic Bank Marketing Dataset (DP-GAN)

In [ ]:
# Assigning file path for importing the datasets.
file_path = "./synthetic-datasets_DP-GAN_B=512/bank-marketing/"

# Computing the metrics for the best performing datasets.
sample_2 = pd.read_csv(file_path + "epsi_1/run1/sample2_encoded.csv")
asf_2, ncf_2 = fairness_analysis(train_set = sample_2, test_set = bank_test_encoded, dataset = "Bank")
print("asf_scores (mean, std): ", (np.mean(asf_2), np.std(asf_2)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_2), np.std(ncf_2)))
print()

sample_5 = pd.read_csv(file_path + "epsi_2/run2/sample2_encoded.csv")
asf_5, ncf_5 = fairness_analysis(train_set = sample_5, test_set = bank_test_encoded, dataset = "Bank")
print("asf_scores (mean, std): ", (np.mean(asf_5), np.std(asf_5)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_5), np.std(ncf_5)))
print()

sample_8 = pd.read_csv(file_path + "epsi_5/run3/sample2_encoded.csv")
asf_8, ncf_8 = fairness_analysis(train_set = sample_8, test_set = bank_test_encoded, dataset = "Bank")
print("asf_scores (mean, std): ", (np.mean(asf_8), np.std(asf_8)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_8), np.std(ncf_8)))
print()

sample_6 = pd.read_csv(file_path + "epsi_8/run2/sample3_encoded.csv")
asf_6, ncf_6 = fairness_analysis(train_set = sample_6, test_set = bank_test_encoded, dataset = "Bank")
print("asf_scores (mean, std): ", (np.mean(asf_6), np.std(asf_6)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_6), np.std(ncf_6)))

bank_results_DP = pd.DataFrame({"sample_2": {"asf": np.mean(asf_2), "ncf": np.mean(ncf_2)}, 
                                "sample_5": {"asf": np.mean(asf_5), "ncf": np.mean(ncf_5)}, 
                                "sample_8": {"asf": np.mean(asf_8), "ncf": np.mean(ncf_8)},
                                "sample_6": {"asf": np.mean(asf_6), "ncf": np.mean(ncf_6)}}).T
bank_results_DP_std = pd.DataFrame({"sample_2": {"asf": np.std(asf_2), "ncf": np.std(ncf_2)}, 
                                    "sample_5": {"asf": np.std(asf_5), "ncf": np.std(ncf_5)}, 
                                    "sample_8": {"asf": np.std(asf_8), "ncf": np.std(ncf_8)},
                                    "sample_6": {"asf": np.std(asf_6), "ncf": np.std(ncf_6)}}).T

bank_results_DP_full = pd.DataFrame({"sample_2": {"asf": asf_2, "ncf": ncf_2}, 
                                     "sample_5": {"asf": asf_5, "ncf": ncf_5}, 
                                     "sample_8": {"asf": asf_8, "ncf": ncf_8},
                                     "sample_6": {"asf": asf_6, "ncf": ncf_6}}).T

In [ ]:
#bank_results_DP_full["asf"].iloc[3]
#bank_results_DP_full["ncf"].iloc[3]
bank_results_DP_full

## Synthetic Bank Marketing Dataset (WGAN-GP)

In [ ]:
# Assigning file path for importing the dataset.
file_path = "./synthetic-datasets_WGAN-GP/bank-marketing/"

# Computing the metrics for the best performing dataset.
sample_13 = pd.read_csv(file_path + "run5/sample1_encoded.csv")
bank_wgangp_asf, bank_wgangp_ncf = fairness_analysis(train_set = sample_13, test_set = bank_test_encoded, dataset = "Bank")
print("asf_scores (mean, std): ", (np.mean(bank_wgangp_asf), np.std(bank_wgangp_asf)))  
print("ncf_scores (mean, std): ", (np.mean(bank_wgangp_ncf), np.std(bank_wgangp_ncf)))

bank_wgangp_results = pd.DataFrame({"asf": (np.mean(bank_wgangp_asf), np.std(bank_wgangp_asf)), 
                                    "ncf": (np.mean(bank_wgangp_ncf), np.std(bank_wgangp_ncf))}).rename(index = {0: "mean", 1: "std"})

In [ ]:
print(bank_wgangp_asf, bank_wgangp_ncf)
bank_wgangp_results

## Credit Card Default Dataset (Real)

In [ ]:
# Individual fairness analysis of Credit Card Default dataset.
credit_asf, credit_ncf = fairness_analysis(train_set = credit_train_encoded, test_set = credit_test_encoded, dataset = "Credit")
print("asf_scores (mean, std): ", (np.mean(credit_asf), np.std(credit_asf)))  
print("ncf_scores (mean, std): ", (np.mean(credit_ncf), np.std(credit_ncf)))

credit_results = pd.DataFrame({"asf": (np.mean(credit_asf), np.std(credit_asf)), 
                               "ncf": (np.mean(credit_ncf), np.std(credit_ncf))}).rename(index = {0: "mean", 1: "std"})

In [ ]:
print(credit_asf, credit_ncf)
credit_results

## Synthetic Credit Card Default Dataset (PATE-GAN)

In [ ]:
# Assigning file path for importing the datasets.
file_path = "./synthetic-datasets_PATE-GAN/credit-card-default/"

# Computing the metrics for the best performing datasets.
sample_9 = pd.read_csv(file_path + "epsi_1/run3/sample3_encoded.csv")
asf_9, ncf_9 = fairness_analysis(train_set = sample_9, test_set = credit_test_encoded, dataset = "Credit")
print("asf_scores (mean, std): ", (np.mean(asf_9), np.std(asf_9)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_9), np.std(ncf_9)))
print()

sample_5 = pd.read_csv(file_path + "epsi_2/run2/sample2_encoded.csv")
asf_5, ncf_5 = fairness_analysis(train_set = sample_5, test_set = credit_test_encoded, dataset = "Credit")
print("asf_scores (mean, std): ", (np.mean(asf_5), np.std(asf_5)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_5), np.std(ncf_5)))
print()

sample_13 = pd.read_csv(file_path + "epsi_5/run5/sample1_encoded.csv")
asf_13, ncf_13 = fairness_analysis(train_set = sample_13, test_set = credit_test_encoded, dataset = "Credit")
print("asf_scores (mean, std): ", (np.mean(asf_13), np.std(asf_13)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_13), np.std(ncf_13)))
print()

sample_1 = pd.read_csv(file_path + "epsi_8/run1/sample1_encoded.csv")
asf_1, ncf_1 = fairness_analysis(train_set = sample_1, test_set = credit_test_encoded, dataset = "Credit")
print("asf_scores (mean, std): ", (np.mean(asf_1), np.std(asf_1)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_1), np.std(ncf_1)))

credit_results_PT = pd.DataFrame({"sample_9": {"asf": np.mean(asf_9), "ncf": np.mean(ncf_9)}, 
                                  "sample_5": {"asf": np.mean(asf_5), "ncf": np.mean(ncf_5)}, 
                                  "sample_13": {"asf": np.mean(asf_13), "ncf": np.mean(ncf_13)},
                                  "sample_1": {"asf": np.mean(asf_1), "ncf": np.mean(ncf_1)}}).T
credit_results_PT_std = pd.DataFrame({"sample_9": {"asf": np.std(asf_9), "ncf": np.std(ncf_9)}, 
                                      "sample_5": {"asf": np.std(asf_5), "ncf": np.std(ncf_5)}, 
                                      "sample_13": {"asf": np.std(asf_13), "ncf": np.std(ncf_13)},
                                      "sample_1": {"asf": np.std(asf_1), "ncf": np.std(ncf_1)}}).T

credit_results_PT_full = pd.DataFrame({"sample_9": {"asf":  asf_9, "ncf": ncf_9}, 
                                       "sample_5": {"asf":  asf_5, "ncf": ncf_5}, 
                                       "sample_13": {"asf":  asf_13, "ncf": ncf_13},
                                       "sample_1": {"asf":  asf_1, "ncf": ncf_1}}).T

In [ ]:
#credit_results_PT_full["asf"].iloc[3]
#credit_results_PT_full["ncf"].iloc[3]
credit_results_PT_full

## Synthetic Credit Card Default Dataset (DP-GAN)

In [ ]:
# Assigning file path for importing the datasets.
file_path = "./synthetic-datasets_DP-GAN_B=512/credit-card-default/"

# Computing the metrics for the best performing datasets.
sample_2 = pd.read_csv(file_path + "epsi_1/run1/sample2_encoded.csv")
asf_2, ncf_2 = fairness_analysis(train_set = sample_2, test_set = credit_test_encoded, dataset = "Credit")
print("asf_scores (mean, std): ", (np.mean(asf_2), np.std(asf_2)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_2), np.std(ncf_2)))
print()

sample_8 = pd.read_csv(file_path + "epsi_2/run3/sample2_encoded.csv")
asf_8, ncf_8 = fairness_analysis(train_set = sample_8, test_set = credit_test_encoded, dataset = "Credit")
print("asf_scores (mean, std): ", (np.mean(asf_8), np.std(asf_8)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_8), np.std(ncf_8)))
print()

sample_4 = pd.read_csv(file_path + "epsi_5/run2/sample1_encoded.csv")
asf_4, ncf_4 = fairness_analysis(train_set = sample_4, test_set = credit_test_encoded, dataset = "Credit")
print("asf_scores (mean, std): ", (np.mean(asf_4), np.std(asf_4)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_4), np.std(ncf_4)))
print()

sample_11 = pd.read_csv(file_path + "epsi_8/run4/sample2_encoded.csv")
asf_11, ncf_11 = fairness_analysis(train_set = sample_11, test_set = credit_test_encoded, dataset = "Credit")
print("asf_scores (mean, std): ", (np.mean(asf_11), np.std(asf_11)))  
print("ncf_scores (mean, std): ", (np.mean(ncf_11), np.std(ncf_11)))

credit_results_DP = pd.DataFrame({"sample_2": {"asf": np.mean(asf_2), "ncf": np.mean(ncf_2)}, 
                                  "sample_8": {"asf": np.mean(asf_8), "ncf": np.mean(ncf_8)}, 
                                  "sample_4": {"asf": np.mean(asf_4), "ncf": np.mean(ncf_4)},
                                  "sample_11": {"asf": np.mean(asf_11), "ncf": np.mean(ncf_11)}}).T
credit_results_DP_std = pd.DataFrame({"sample_2": {"asf": np.std(asf_2), "ncf": np.std(ncf_2)}, 
                                      "sample_8": {"asf": np.std(asf_8), "ncf": np.std(ncf_8)}, 
                                      "sample_4": {"asf": np.std(asf_4), "ncf": np.std(ncf_4)},
                                      "sample_11": {"asf": np.std(asf_11), "ncf": np.std(ncf_11)}}).T

credit_results_DP_full = pd.DataFrame({"sample_2": {"asf": asf_2, "ncf": ncf_2}, 
                                       "sample_8": {"asf": asf_8, "ncf": ncf_8}, 
                                       "sample_4": {"asf": asf_4, "ncf": ncf_4},
                                       "sample_11": {"asf": asf_11, "ncf": ncf_11}}).T

In [ ]:
#credit_results_DP_full["asf"].iloc[3]
#credit_results_DP_full["ncf"].iloc[3]
credit_results_DP_full

## Synthetic Credit Card Default Dataset (WGAN-GP)

In [ ]:
# Assigning file path for importing the dataset.
file_path = "./synthetic-datasets_WGAN-GP/credit-card-default/"

# Computing the metrics for the best performing dataset.
sample_8 = pd.read_csv(file_path + "run3/sample2_encoded.csv")
credit_wgangp_asf, credit_wgangp_ncf = fairness_analysis(train_set = sample_8, test_set = credit_test_encoded, dataset = "Credit")
print("asf_scores (mean, std): ", (np.mean(credit_wgangp_asf), np.std(credit_wgangp_asf)))  
print("ncf_scores (mean, std): ", (np.mean(credit_wgangp_ncf), np.std(credit_wgangp_ncf)))

credit_wgangp_results = pd.DataFrame({"asf": (np.mean(credit_wgangp_asf), np.std(credit_wgangp_asf)), 
                                      "ncf": (np.mean(credit_wgangp_ncf), np.std(credit_wgangp_ncf))}).rename(index = {0: "mean", 1: "std"})

In [ ]:
print(credit_wgangp_asf, credit_wgangp_ncf)
credit_wgangp_results

## Average of Results

In [ ]:
# Average of datasets' similarity fairness results.
avg_asf_real = [adult_asf, bank_asf, credit_asf]
avg_asf_wgan = [adult_wgangp_asf, bank_wgangp_asf, credit_wgangp_asf]
avg_asf_pate = [adult_results_PT_full["asf"][e] + bank_results_PT_full["asf"][e] + credit_results_PT_full["asf"][e] for e in range(0, 4)]
avg_asf_dp = [adult_results_DP_full["asf"][e] + bank_results_DP_full["asf"][e] + credit_results_DP_full["asf"][e] for e in range(0, 4)]

# Average of datasets' neighborhood consistency fairness results.
avg_ncf_real = [adult_ncf, bank_ncf, credit_ncf]
avg_ncf_wgan = [adult_wgangp_ncf, bank_wgangp_ncf, credit_wgangp_ncf]
avg_ncf_pate = [adult_results_PT_full["ncf"][e] + bank_results_PT_full["ncf"][e] + credit_results_PT_full["ncf"][e] for e in range(0, 4)]
avg_ncf_dp = [adult_results_DP_full["ncf"][e] + bank_results_DP_full["ncf"][e] + credit_results_DP_full["ncf"][e] for e in range(0, 4)]

## Plots of Results

In [ ]:
# Plotting individual fairness metrics of synthetic datasets (PATE-GAN).
colors = ["red", "blue", "olive"]
labels = ["Original", "WGAN-GP", "PATE-GAN"]
epsi = [1, 2, 5, 8]
msize = 8 

fairness_clf = ["asf", "ncf"]
fairness_labels = ["ASF", "ANCF"]

for f in fairness_clf:
    # Create a figure for the aggregated plot.
    fig, axes = plt.subplots(1, 4, figsize=(16, 4.0))  # 1 x 4 grid
    for c in range(0, 4):
        if c == 0:
            dataset = "Adult"
            results_np = pd.concat([adult_results[f], adult_wgangp_results[f]], axis = 1).set_axis(["Real", "WGAN-GP"], axis = 1)
            results_dp = pd.concat([adult_results_PT[f], adult_results_PT_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results = pd.concat([results_np, results_dp], axis = 1)
        elif c == 1:
            dataset = "Bank"
            results_np = pd.concat([bank_results[f], bank_wgangp_results[f]], axis = 1).set_axis(["Real", "WGAN-GP"], axis = 1)
            results_dp = pd.concat([bank_results_PT[f], bank_results_PT_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results = pd.concat([results_np, results_dp], axis = 1)
        elif c == 2:
            dataset = "Credit"
            results_np = pd.concat([credit_results[f], credit_wgangp_results[f]], axis = 1).set_axis(["Real", "WGAN-GP"], axis = 1)
            results_dp = pd.concat([credit_results_PT[f], credit_results_PT_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results = pd.concat([results_np, results_dp], axis = 1)
        else:
            dataset = "Average"
            if f == "asf":
                results = pd.DataFrame({"mean": [np.mean(avg_asf_real), np.mean(avg_asf_wgan)] + list(np.mean(avg_asf_pate, axis = 1)),
                                        "std": [np.std(avg_asf_real), np.std(avg_asf_wgan)] + list(np.std(avg_asf_pate, axis = 1))},
                                       index = ["Real", "WGAN-GP", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
            else: 
                results = pd.DataFrame({"mean": [np.mean(avg_ncf_real), np.mean(avg_ncf_wgan)] + list(np.mean(avg_ncf_pate, axis = 1)), 
                                        "std": [np.std(avg_ncf_real), np.std(avg_ncf_wgan)] + list(np.std(avg_ncf_pate, axis = 1))},
                                       index = ["Real", "WGAN-GP", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
            
    
        col = fairness_clf.index(f)
        ax = axes[c]

        # Plotting a dashed line for the individual fairness metrics.
        ax.axhline(0, 0, 5, ls = "--", c = "black")
    
        # Create line plots for the current metric: Adult, Bank, Credit & Average.            
        ax.plot([0, 10], [results.iloc[0, 1]]*len([0, 10]), color = colors[1])   # WGAN-GP
        ax.fill_between([0, 10], [results.iloc[0, 1] + results.iloc[1, 1]]*len([0, 10]),
                        [results.iloc[0, 1] - results.iloc[1, 1]]*len([0, 10]), alpha = 0.1, color = colors[1])

        ax.plot([0, 10], [results.iloc[0, 0]]*len([0, 10]), color = colors[0])   # original
        ax.fill_between([0, 10], [results.iloc[0, 0] + results.iloc[1, 0]]*len([0, 10]),
                        [results.iloc[0, 0] - results.iloc[1, 0]]*len([0, 10]), alpha = 0.1, color = colors[0])
        
        ax.errorbar(epsi, results.iloc[0, 2:], yerr = results.iloc[1, 2:], capsize = 2, color = colors[2], marker = "o",   # PATE-GAN
                    markersize = msize)
            
        # Customizing the plot.
        if c == 0:
            ax.set_ylabel(fairness_labels[col])
        ax.set_aspect('auto')
        ax.set_xlim([0.6, 8.6])
        ax.set_title(f"{dataset}")
        ax.set_xlabel(r"$\epsilon$")
        if f == "asf": 
            ax.set_ylim(-0.03, 0.38)
        else:
            ax.set_ylim(-0.03, 0.31)
        ax.grid(True, linestyle='--', alpha = 0.6)

        if f == "asf":
            patches = [mpatches.Patch(color = colors[i], label = labels[i]) for i in range(len(labels))]
            plt.figlegend(handles = patches, ncol = 3, loc = "upper center", bbox_to_anchor = (0.5, 1.05))
        
        plt.tight_layout(rect = [0, 0, 1, 0.95])
    
    plt.show()

In [ ]:
# Plotting individual fairness metrics of synthetic datasets (DP-GAN).
colors = ["red", "blue", "purple"]
labels = ["Original", "WGAN-GP", "DP-GAN"]
epsi = [1, 2, 5, 8]
msize = 8 

fairness_clf = ["asf", "ncf"]
fairness_labels = ["ASF", "ANCF"]

for f in fairness_clf:
    # Create a figure for the aggregated plot.
    fig, axes = plt.subplots(1, 4, figsize=(16, 4.0))  # 1 x 4 grid
    for c in range(0, 4):
        if c == 0:
            dataset = "Adult"
            results_np = pd.concat([adult_results[f], adult_wgangp_results[f]], axis = 1).set_axis(["Real", "WGAN-GP"], axis = 1)
            results_dp = pd.concat([adult_results_DP[f], adult_results_DP_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results = pd.concat([results_np, results_dp], axis = 1)
        elif c == 1:
            dataset = "Bank"
            results_np = pd.concat([bank_results[f], bank_wgangp_results[f]], axis = 1).set_axis(["Real", "WGAN-GP"], axis = 1)
            results_dp = pd.concat([bank_results_DP[f], bank_results_DP_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results = pd.concat([results_np, results_dp], axis = 1)
        elif c == 2:
            dataset = "Credit"
            results_np = pd.concat([credit_results[f], credit_wgangp_results[f]], axis = 1).set_axis(["Real", "WGAN-GP"], axis = 1)
            results_dp = pd.concat([credit_results_DP[f], credit_results_DP_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results = pd.concat([results_np, results_dp], axis = 1)
        else:
            dataset = "Average"
            if f == "asf":
                results = pd.DataFrame({"mean": [np.mean(avg_asf_real), np.mean(avg_asf_wgan)] + list(np.mean(avg_asf_dp, axis = 1)),
                                        "std": [np.std(avg_asf_real), np.std(avg_asf_wgan)] + list(np.std(avg_asf_dp, axis = 1))},
                                       index = ["Real", "WGAN-GP", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
            else: 
                results = pd.DataFrame({"mean": [np.mean(avg_ncf_real), np.mean(avg_ncf_wgan)] + list(np.mean(avg_ncf_dp, axis = 1)), 
                                        "std": [np.std(avg_ncf_real), np.std(avg_ncf_wgan)] + list(np.std(avg_ncf_dp, axis = 1))},
                                       index = ["Real", "WGAN-GP", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
            
    
        col = fairness_clf.index(f)
        ax = axes[c]

        # Plotting a dashed line for the individual fairness metrics.
        ax.axhline(0, 0, 5, ls = "--", c = "black")
    
        # Create line plots for the current metric: Adult, Bank, Credit & Average.     
        ax.plot([0, 10], [results.iloc[0, 1]]*len([0, 10]), color = colors[1])   # WGAN-GP
        ax.fill_between([0, 10], [results.iloc[0, 1] + results.iloc[1, 1]]*len([0, 10]),
                        [results.iloc[0, 1] - results.iloc[1, 1]]*len([0, 10]), alpha = 0.1, color = colors[1])

        ax.plot([0, 10], [results.iloc[0, 0]]*len([0, 10]), color = colors[0])   # original
        ax.fill_between([0, 10], [results.iloc[0, 0] + results.iloc[1, 0]]*len([0, 10]),
                        [results.iloc[0, 0] - results.iloc[1, 0]]*len([0, 10]), alpha = 0.1, color = colors[0])
        
        ax.errorbar(epsi, results.iloc[0, 2:], yerr = results.iloc[1, 2:], capsize = 2, color = colors[2], marker = "o",   # DP-GAN
                    markersize = msize)
            
        # Customizing the plot.
        if c == 0:
            ax.set_ylabel(fairness_labels[col])
        ax.set_aspect('auto')
        ax.set_xlim([0.6, 8.6])
        ax.set_title(f"{dataset}")
        ax.set_xlabel(r"$\epsilon$")
        if f == "asf": 
            ax.set_ylim(-0.03, 0.46)
        else:
            ax.set_ylim(-0.03, 0.38)
        ax.grid(True, linestyle='--', alpha = 0.6)

        if f == "asf":
            patches = [mpatches.Patch(color = colors[i], label = labels[i]) for i in range(len(labels))]
            plt.figlegend(handles = patches, ncol = 3, loc = "upper center", bbox_to_anchor = (0.5, 1.05))
        
        plt.tight_layout(rect = [0, 0, 1, 0.95])
    
    plt.show()